# 5th attempt - RNN

In [25]:
# 1️⃣ ייבוא המודול והפונקציות
import functions         # ייבוא רגיל למודול
from functions import *  # ייבוא כל הפונקציות ל־namespace

# 2️⃣ טעינה מחדש
import importlib
importlib.reload(functions)  # מוודא שהמודול מעודכן

# 3️⃣ הרצת הפונקציות שוב ל־namespace
from functions import *  # שוב מייבא את כל הפונקציות אחרי ה־reload

In [26]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [27]:
SIZE = 5
AMOUNT_BOARDS = 10000  # 100000 file is corrupted, using 1000
AMOUNT_MOVES = 100
NUM_DICT = 1
gen = 2

In [28]:
reverse_df_sort = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=0, test_size=0.1, val_size=0.1, random_state=365)
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE, gen)

print("len df:", len(reverse_df_sort))
print("len train: ", len(X_train))
print("len val: ",len(X_val))
print("len test: ",len(X_test))

len df: 29760
len train:  24105
len val:  2679
len test:  2976


In [29]:
from functions import build_and_train_rnn

# build and train RNN model
model, history = build_and_train_rnn(
    X_train_array,
    y_train_array,
    size=SIZE,
    gen=gen,
    rnn_units=128,
    dense_units=64,
    epochs=20,
    batch_size=512,
    validation_split=0.2
)

model.summary()

Epoch 1/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.5635 - loss: 0.6852 - val_accuracy: 0.6295 - val_loss: 0.6746 - learning_rate: 0.0010
Epoch 2/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6605 - loss: 0.6543 - val_accuracy: 0.6727 - val_loss: 0.6393 - learning_rate: 0.0010
Epoch 3/20
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6750 - loss: 0.6462
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.6749 - loss: 0.6463 - val_accuracy: 0.6741 - val_loss: 0.6574 - learning_rate: 0.0010
Epoch 4/20
34/38 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6739 - loss: 0.6457
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6741 - loss: 0.6456 - val_accuracy: 0.6725 - val_loss: 0.6470 - learning_rate: 5.0000e-04
Epoch 5/20
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6757 - loss: 0.6423
Epoch 5:

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        78,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 261,509 (1021.52 KB)

 Trainable params: 87,169 (340.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 174,340 (681.02 KB)

In [30]:
from functions import build_and_train_rnn_v2

# build and train IMPROVED RNN model (Bidirectional stacked LSTM)
model_v2, history_v2 = build_and_train_rnn_v2(
    X_train_array,
    y_train_array,
    size=SIZE,
    gen=gen,
    lstm_units_1=128,
    lstm_units_2=64,
    dense_units_1=128,
    dense_units_2=64,
    dropout_rate=0.3,
    recurrent_dropout=0.2,
    learning_rate=0.001,
    epochs=40,
    batch_size=512,
    validation_split=0.2
)

model_v2.summary()

Epoch 1/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 27s 197ms/step - accuracy: 0.5655 - loss: 0.7947 - val_accuracy: 0.3408 - val_loss: 0.6960 - learning_rate: 0.0010
Epoch 2/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.5917 - loss: 0.6749 - val_accuracy: 0.3408 - val_loss: 0.6951 - learning_rate: 0.0010
Epoch 3/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.6404 - loss: 0.6584 - val_accuracy: 0.3470 - val_loss: 0.6952 - learning_rate: 0.0010
Epoch 4/40
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6442 - loss: 0.6485
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.6442 - loss: 0.6485 - val_accuracy: 0.3626 - val_loss: 0.6957 - learning_rate: 0.0010
Epoch 5/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - accuracy: 0.6486 - loss: 0.6411 - val_accuracy: 0.4485 - val_loss: 0.6937 - learning_rate: 5.0000e-04
Epoch 6/40
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.6494 - loss: 0.6366 - va

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 1, 256)         │       157,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 1, 256)         │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,043,717 (3.98 MB)

 Trainable params: 347,649 (1.33 MB)

 Non-trainable params: 768 (3.00 KB)

 Optimizer params: 695,300 (2.65 MB)

In [31]:
# --- Optuna hyperparameter search for CRNN/ConvLSTM ---
import importlib
import optuna_search
importlib.reload(optuna_search)
from optuna_search import run_optuna_search

study = run_optuna_search(
    X_train_array, y_train_array,
    X_val_array, y_val_array,
    size=SIZE, gen=gen,
    n_trials=50
)

# Show top-5 trials
print("\nTop 5 trials:")
for t in sorted(study.trials, key=lambda t: t.value or 0, reverse=True)[:5]:
    print(f"  Trial {t.number}: val_acc={t.value:.4f}  params={t.params}")

[I 2026-03-30 07:25:07,553] A new study created in memory with name: gol_crnn_search
  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.71034:   2%|▏         | 1/50 [01:15<1:01:30, 75.31s/it]

[I 2026-03-30 07:26:22,867] Trial 0 finished with value: 0.7103396654129028 and parameters: {'n_conv_layers': 1, 'conv_filters': 64, 'rnn_type': 'LSTM', 'rnn_units': 128, 'dropout_rate': 0.30000000000000004, 'learning_rate': 0.0018767252034712749, 'use_bidirectional': True, 'dense_units': 64, 'batch_size': 512}. Best is trial 0 with value: 0.7103396654129028.


Best trial: 0. Best value: 0.71034:   4%|▍         | 2/50 [03:28<1:27:30, 109.38s/it]

[I 2026-03-30 07:28:36,105] Trial 1 finished with value: 0.7081000208854675 and parameters: {'n_conv_layers': 2, 'conv_filters': 64, 'rnn_type': 'GRU', 'rnn_units': 64, 'dropout_rate': 0.35, 'learning_rate': 0.0002822608688936417, 'use_bidirectional': False, 'dense_units': 32, 'batch_size': 256}. Best is trial 0 with value: 0.7103396654129028.


...

[I 2026-03-30 08:02:23,170] Trial 47 finished with value: 0.7136991620063782 and parameters: {'n_conv_layers': 2, 'conv_filters': 16, 'rnn_type': 'LSTM', 'rnn_units': 64, 'dropout_rate': 0.1, 'learning_rate': 0.0009850122103522956, 'use_bidirectional': True, 'dense_units': 128, 'batch_size': 256}. Best is trial 12 with value: 0.717431902885437.


Best trial: 12. Best value: 0.717432:  98%|█████████▊| 49/50 [37:54<00:37, 38.00s/it]

[I 2026-03-30 08:03:01,682] Trial 48 pruned. 


Best trial: 12. Best value: 0.717432: 100%|██████████| 50/50 [38:25<00:00, 46.11s/it]

[I 2026-03-30 08:03:32,944] Trial 49 pruned. 

OPTUNA SEARCH COMPLETE
Best trial #12
  Val accuracy : 0.7174
  Params       : {'n_conv_layers': 1, 'conv_filters': 64, 'rnn_type': 'LSTM', 'rnn_units': 128, 'dropout_rate': 0.1, 'learning_rate': 0.000612802946090018, 'use_bidirectional': True, 'dense_units': 64, 'batch_size': 256}

Top 5 trials:
  Trial 12: val_acc=0.7174  params={'n_conv_layers': 1, 'conv_filters': 64, 'rnn_type': 'LSTM', 'rnn_units': 128, 'dropout_rate': 0.1, 'learning_rate': 0.000612802946090018, 'use_bidirectional': True, 'dense_units': 64, 'batch_size': 256}
  Trial 45: val_acc=0.7159  params={'n_conv_layers': 2, 'conv_filters': 64, 'rnn_type': 'GRU', 'rnn_units': 128, 'dropout_rate': 0.1, 'learning_rate': 0.004695312038301214, 'use_bidirectional': False, 'dense_units': 128, 'batch_size': 256}
  Trial 3: val_acc=0.7152  params={'n_conv_layers': 2, 'conv_filters': 64, 'rnn_type': 'LSTM', 'rnn_units': 64, 'dropout_rate': 0.45000000000000007, 'learning_rate': 0.00522435

In [32]:
from functions import build_and_train_crnn_v3

# build and train CRNN model (Conv2D + Bidirectional stacked LSTM + AdamW)
model_v3, history_v3 = build_and_train_crnn_v3(
    X_train_array,
    y_train_array,
    size=SIZE,
    gen=gen,
    conv_filters=(32, 64),
    lstm_units_1=128,
    lstm_units_2=64,
    dense_units_1=128,
    dense_units_2=64,
    dropout_rate=0.3,
    recurrent_dropout=0.2,
    learning_rate=0.001,
    weight_decay=1e-4,
    epochs=50,
    batch_size=512,
    validation_split=0.2
)

model_v3.summary()

Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 15s 99ms/step - accuracy: 0.4859 - loss: 0.7758 - val_accuracy: 0.4022 - val_loss: 0.6935 - learning_rate: 0.0010
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - accuracy: 0.5372 - loss: 0.7061 - val_accuracy: 0.3408 - val_loss: 0.6963 - learning_rate: 0.0010
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.5809 - loss: 0.6840 - val_accuracy: 0.6754 - val_loss: 0.6889 - learning_rate: 0.0010
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 67ms/step - accuracy: 0.6080 - loss: 0.6659 - val_accuracy: 0.6609 - val_loss: 0.6804 - learning_rate: 0.0010
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 67ms/step - accuracy: 0.6218 - loss: 0.6574 - val_accuracy: 0.6592 - val_loss: 0.6670 - learning_rate: 0.0010
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.6206 - loss: 0.6568 - val_accuracy: 0.6839 - val_loss: 0.6693 - learning_rate: 0.0010
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.6448 - loss: 0.6462 - val_ac

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_7              │ (None, 1, 5, 5, 32)    │           320 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_8              │ (None, 1, 5, 5, 32)    │           128 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_9              │ (None, 1, 5, 5, 64)    │        18,496 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_10             │ (None, 1, 5, 5, 64)    │           256 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_11             │ (None, 1, 2, 2, 64)    │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_12             │ (None, 1, 256)         │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_13             │ (None, 1, 256)         │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 1, 256)         │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 1, 256)         │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,811,589 (6.91 MB)

 Trainable params: 603,457 (2.30 MB)

 Non-trainable params: 1,216 (4.75 KB)

 Optimizer params: 1,206,916 (4.60 MB)

In [33]:
# Evaluate CRNN v3
X_test_crnn = X_test_array.reshape((-1, gen-1, SIZE, SIZE, 1)).astype('float32')

test_loss_v3, test_acc_v3 = model_v3.evaluate(X_test_crnn, y_test_array.astype('float32'))
print(f"Test accuracy (v3 CRNN): {test_acc_v3:.3f}")

evaluate_model(model_v3, X_test_crnn, y_test_array, SIZE, gen)

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6583 - loss: 0.6269
Test accuracy (v3 CRNN): 0.664
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        459 │        562 │
│ True=Dead    │        438 │       1517 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.664
Precision   : 0.512
Recall      : 0.450
F1-score    : 0.479


In [34]:
X_test_rnn_v2 = X_test_array.reshape((-1, gen-1, SIZE*SIZE)).astype('float32')

test_loss_v2, test_acc_v2 = model_v2.evaluate(X_test_rnn_v2, y_test_array.astype('float32'))
print(f"Test accuracy (v2): {test_acc_v2:.3f}")

93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6328 - loss: 0.6436
Test accuracy (v2): 0.640


In [35]:
evaluate_model(model_v2, X_test_rnn_v2, y_test_array, SIZE, gen)

93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        619 │        402 │
│ True=Dead    │        668 │       1287 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.640
Precision   : 0.481
Recall      : 0.606
F1-score    : 0.536


In [36]:
X_test_rnn = X_test_array.reshape((-1, gen-1, SIZE*SIZE)).astype('float32')

test_loss, test_acc = model.evaluate(X_test_rnn, y_test_array.astype('float32'))
print(f"Test accuracy: {test_acc:.3f}")

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6530 - loss: 0.6493
Test accuracy: 0.659


In [37]:
evaluate_model(model, X_test_rnn, y_test_array, SIZE, gen)

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        491 │        530 │
│ True=Dead    │        485 │       1470 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.659
Precision   : 0.503
Recall      : 0.481
F1-score    : 0.492


In [38]:
# Quick diagnostic: data shapes and current model performance
print(f"SIZE={SIZE}, gen={gen}, AMOUNT_BOARDS={AMOUNT_BOARDS}")
print(f"X_train_array shape: {X_train_array.shape}")
print(f"X_val_array shape:   {X_val_array.shape}")
print(f"X_test_array shape:  {X_test_array.shape}")
print(f"y_train_array shape: {y_train_array.shape}")
print(f"reverse_df_sort shape: {reverse_df_sort.shape}")
print(f"\nClass balance (y_train): {np.mean(y_train_array):.3f}")

# Check existing model performance
X_test_rnn = X_test_array.reshape((-1, gen-1, SIZE*SIZE)).astype('float32')
test_loss, test_acc = model.evaluate(X_test_rnn, y_test_array.astype('float32'), verbose=0)
print(f"\nCurrent RNN (v1) test accuracy: {test_acc:.4f}")

SIZE=5, gen=2, AMOUNT_BOARDS=10000
X_train_array shape: (24105, 5, 5, 1)
X_val_array shape:   (2679, 5, 5, 1)
X_test_array shape:  (2976, 5, 5, 1)
y_train_array shape: (24105, 1)
reverse_df_sort shape: (29760, 50)

Class balance (y_train): 0.347

Current RNN (v1) test accuracy: 0.6589


In [ ]:
# ============================================================
# ITERATION 1: Residual FCN + Wrap Padding + Full-Board Prediction
# ============================================================
# KEY INSIGHTS:
# - gen=2 → timesteps=1 → RNN is useless, this is a SPATIAL problem
# - GoL board is TOROIDAL → need wrap padding for CNN
# - Class imbalance (20.5% alive) → current 0.80 ≈ "predict all dead"
# - Predict ALL 100 pixels → 100x more gradient signal
# ============================================================
import tensorflow as tf
import numpy as np
tf.keras.backend.clear_session()

# --- 1) Extract full-board targets using same split indices ---
amount_feat = (gen - 1) * SIZE * SIZE  # 100
target_cols = [f'Col_{amount_feat + i}' for i in range(SIZE * SIZE)]

y_train_board = reverse_df_sort.loc[X_train.index, target_cols].to_numpy().reshape(-1, SIZE, SIZE, 1).astype('float32')
y_val_board   = reverse_df_sort.loc[X_val.index, target_cols].to_numpy().reshape(-1, SIZE, SIZE, 1).astype('float32')
y_test_board  = reverse_df_sort.loc[X_test.index, target_cols].to_numpy().reshape(-1, SIZE, SIZE, 1).astype('float32')

# --- 2) Wrap-pad inputs for toroidal topology ---
PAD = 3
X_train_wp = np.pad(X_train_array, ((0,0),(PAD,PAD),(PAD,PAD),(0,0)), mode='wrap').astype('float32')
X_val_wp   = np.pad(X_val_array,   ((0,0),(PAD,PAD),(PAD,PAD),(0,0)), mode='wrap').astype('float32')
X_test_wp  = np.pad(X_test_array,  ((0,0),(PAD,PAD),(PAD,PAD),(0,0)), mode='wrap').astype('float32')

print(f"Input:  {X_train_wp.shape}")    # (n, 16, 16, 1)
print(f"Target: {y_train_board.shape}")  # (n, 10, 10, 1)
print(f"Target alive fraction: {np.mean(y_train_board):.3f}")

# --- 3) Compute class weight for binary pixels ---
frac_alive = float(np.mean(y_train_board))
w1 = (1 - frac_alive) / frac_alive  # weight for alive class
print(f"Alive weight: {w1:.2f}")

# --- 4) Build Residual FCN ---
padded_size = SIZE + 2 * PAD  # 16
inputs = tf.keras.Input(shape=(padded_size, padded_size, 1))

# Initial conv
x = tf.keras.layers.Conv2D(64, 3, padding='same')(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Activation('relu')(x)

# Residual block helper
def res_block(x, filters, downsample=False):
    shortcut = x
    if downsample or x.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, padding='same')(shortcut)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation('relu')(x)
    return x

x = res_block(x, 64)
x = res_block(x, 128)
x = res_block(x, 128)
x = res_block(x, 64)

# Output: 1x1 conv → sigmoid → crop to 10x10
x = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(x)
x = tf.keras.layers.Cropping2D(cropping=((PAD, PAD), (PAD, PAD)))(x)

model_iter1 = tf.keras.Model(inputs, x, name='ResFCN_WrapPad')

# Weighted binary crossentropy
def weighted_bce(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    weights = y_true * w1 + (1 - y_true) * 1.0  # alive gets higher weight
    weights = tf.squeeze(weights, axis=-1)
    return bce * weights

model_iter1.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss=weighted_bce,
    metrics=['accuracy']
)

model_iter1.summary()

# --- 5) Train ---
callbacks_iter1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

history_iter1 = model_iter1.fit(
    X_train_wp, y_train_board,
    validation_data=(X_val_wp, y_val_board),
    epochs=60,
    batch_size=256,
    callbacks=callbacks_iter1,
    verbose=2
)

# --- 6) Evaluate ---
best_val_acc = max(history_iter1.history['val_accuracy'])
print(f"\n{'='*50}")
print(f"Best val accuracy (all pixels): {best_val_acc:.4f}")

y_pred_board = model_iter1.predict(X_test_wp, verbose=0)
all_pred = (y_pred_board > 0.5).astype(int)
all_true = y_test_board.astype(int)
overall_acc = np.mean(all_pred == all_true)
print(f"Test accuracy (all pixels):     {overall_acc:.4f}")

# Pixel-0 specific
p0_pred = (y_pred_board[:, 0, 0, 0] > 0.5).astype(int)
p0_true = y_test_board[:, 0, 0, 0].astype(int)
p0_acc = np.mean(p0_pred == p0_true)
print(f"Pixel-0 test accuracy:          {p0_acc:.4f}")

# F1 for pixel 0
from sklearn.metrics import f1_score, precision_score, recall_score
p0_f1 = f1_score(p0_true, p0_pred)
p0_prec = precision_score(p0_true, p0_pred, zero_division=0)
p0_recall = recall_score(p0_true, p0_pred, zero_division=0)
print(f"Pixel-0 F1={p0_f1:.4f}  Prec={p0_prec:.4f}  Recall={p0_recall:.4f}")

# Per-class accuracy
alive_mask = all_true.flatten() == 1
dead_mask = all_true.flatten() == 0
print(f"Accuracy on ALIVE cells: {np.mean(all_pred.flatten()[alive_mask] == 1):.4f}")
print(f"Accuracy on DEAD cells:  {np.mean(all_pred.flatten()[dead_mask] == 0):.4f}")
print(f"{'='*50}")

Input:  (24105, 11, 11, 1)
Target: (24105, 5, 5, 1)
Target alive fraction: 0.344
Alive weight: 1.90


Model: "ResFCN_WrapPad"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 11, 11, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 11, 11,    │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 11, 11,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 11, 11,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 11, 11,    │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 11, 11,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 11, 11,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 11, 11,    │     36,928 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 11, 11,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 11, 11,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 11, 11,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 11, 11,    │     73,856 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 11, 11,    │        512 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 11, 11,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 11, 11,    │    147,584 │ activation_3[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 11, 11,    │      8,320 │ activation_2[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 11, 11,    │        512 │ conv2d_5[0][0]  

 Total params: 722,561 (2.76 MB)

 Trainable params: 720,513 (2.75 MB)

 Non-trainable params: 2,048 (8.00 KB)

Epoch 1/60
95/95 - 115s - 1s/step - accuracy: 0.6914 - loss: 0.7760 - val_accuracy: 0.6063 - val_loss: 0.9093 - learning_rate: 0.0010
Epoch 2/60
95/95 - 138s - 1s/step - accuracy: 0.7549 - loss: 0.6578 - val_accuracy: 0.6256 - val_loss: 0.9158 - learning_rate: 0.0010
Epoch 3/60
95/95 - 105s - 1s/step - accuracy: 0.7894 - loss: 0.5889 - val_accuracy: 0.6424 - val_loss: 1.0371 - learning_rate: 0.0010
Epoch 4/60

Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
95/95 - 108s - 1s/step - accuracy: 0.8180 - loss: 0.5262 - val_accuracy: 0.7096 - val_loss: 1.0026 - learning_rate: 0.0010
Epoch 5/60
95/95 - 108s - 1s/step - accuracy: 0.8588 - loss: 0.4279 - val_accuracy: 0.7766 - val_loss: 0.6643 - learning_rate: 5.0000e-04
Epoch 6/60
95/95 - 107s - 1s/step - accuracy: 0.8763 - loss: 0.3798 - val_accuracy: 0.8152 - val_loss: 0.5833 - learning_rate: 5.0000e-04
Epoch 7/60
95/95 - 107s - 1s/step - accuracy: 0.8896 - loss: 0.3415 - val_accuracy: 0.8265 - val_loss: 0.5491 -

In [ ]:
# ============================================================
# ITERATION 2: Multi-output MLP with engineered GoL features
# ============================================================
# Iter 1 CNN reached 0.88 val_acc but ~55 min/epoch (CPU-bound).
# Strategy: 
#   - Same full-board prediction (predict all 100 pixels → 100x signal)
#   - Add GoL-specific features: neighbor counts (the core GoL info)
#   - MLP trains in SECONDS per epoch (just matrix multiplies)
# ============================================================
import tensorflow as tf
import numpy as np

# --- 1) Prepare full-board targets (recompute to be safe) ---
amount_feat = (gen - 1) * SIZE * SIZE
target_cols = [f'Col_{amount_feat + i}' for i in range(SIZE * SIZE)]
y_train_board = reverse_df_sort.loc[X_train.index, target_cols].to_numpy().reshape(-1, SIZE*SIZE).astype('float32')
y_val_board   = reverse_df_sort.loc[X_val.index, target_cols].to_numpy().reshape(-1, SIZE*SIZE).astype('float32')
y_test_board  = reverse_df_sort.loc[X_test.index, target_cols].to_numpy().reshape(-1, SIZE*SIZE).astype('float32')

# --- 2) Engineer GoL features: board + wrap-around neighbor counts ---
def gol_features(boards_4d, size=10):
    """Board values (100) + normalized neighbor counts (100) = 200 features."""
    b = boards_4d[:, :, :, 0].astype('float32')       # (n, 10, 10)
    n = b.shape[0]
    padded = np.pad(b, ((0,0),(1,1),(1,1)), mode='wrap')  # toroidal padding
    neigh = np.zeros_like(b)
    for di in [-1, 0, 1]:
        for dj in [-1, 0, 1]:
            if di == 0 and dj == 0:
                continue
            neigh += padded[:, 1+di:size+1+di, 1+dj:size+1+dj]
    return np.concatenate([b.reshape(n,-1), neigh.reshape(n,-1) / 8.0], axis=1)

X_tr = gol_features(X_train_array, SIZE)
X_va = gol_features(X_val_array, SIZE)
X_te = gol_features(X_test_array, SIZE)
print(f"Feature shape: {X_tr.shape}, Target shape: {y_train_board.shape}")

# --- 3) Build MLP ---
tf.keras.backend.clear_session()

model_mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_tr.shape[1],)),  # 200
    tf.keras.layers.Dense(1024, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(SIZE*SIZE, activation='sigmoid'),
], name='MLP_GoLFeatures')

model_mlp.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model_mlp.summary()

# --- 4) Train ---
hist_mlp = model_mlp.fit(
    X_tr, y_train_board,
    validation_data=(X_va, y_val_board),
    epochs=100,
    batch_size=1024,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=12,
                                         restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=5, min_lr=1e-6, verbose=1),
    ],
    verbose=2
)

# --- 5) Evaluate ---
best_val = max(hist_mlp.history['val_accuracy'])
print(f"\n{'='*50}")
print(f"ITERATION 2 RESULTS")
print(f"Best val accuracy (all pixels): {best_val:.4f}")

y_pred = model_mlp.predict(X_te, verbose=0)
pred = (y_pred > 0.5).astype(int)
true = y_test_board.astype(int)

overall = np.mean(pred == true)
p0_pred = pred[:, 0]
p0_true = true[:, 0]
p0_acc = np.mean(p0_pred == p0_true)

from sklearn.metrics import f1_score, precision_score, recall_score
p0_f1 = f1_score(p0_true, p0_pred)
p0_prec = precision_score(p0_true, p0_pred, zero_division=0)
p0_rec = recall_score(p0_true, p0_pred, zero_division=0)

print(f"Test accuracy (all pixels): {overall:.4f}")
print(f"Pixel-0 test acc:           {p0_acc:.4f}")
print(f"Pixel-0 F1={p0_f1:.4f}  Prec={p0_prec:.4f}  Recall={p0_rec:.4f}")

alive_mask = true.flatten() == 1
dead_mask = true.flatten() == 0
print(f"Accuracy on ALIVE cells: {np.mean(pred.flatten()[alive_mask] == 1):.4f}")
print(f"Accuracy on DEAD cells:  {np.mean(pred.flatten()[dead_mask] == 0):.4f}")
print(f"{'='*50}")

# Check if target reached
if best_val >= 0.90:
    print(">>> TARGET 0.90 REACHED! <<<")
    model_mlp.save("models/MLP_GoL_best.keras")
    print("Model saved to models/MLP_GoL_best.keras")